# SHOTpy Tutorial: Using SHOT from Python

This notebook provides a comprehensive guide to using **SHOTpy**, the Python interface for the **SHOT** (Supporting Hyperplane Optimization Toolkit) solver.

SHOT is a solver for mixed-integer nonlinear programming (MINLP) problems that uses a combination of polyhedral outer approximation, extended supporting hyperplanes, and other techniques.

## Table of Contents
1. [Getting Started](#getting-started)
2. [Creating Variables](#creating-variables)
3. [Linear Problems](#linear-problems)
4. [Mixed-Integer Problems](#mixed-integer-problems)
5. [Quadratic Problems](#quadratic-problems)
6. [Nonlinear Problems](#nonlinear-problems)
7. [Reading Problems from Files](#reading-from-files)
8. [Configuring Solver Settings](#solver-settings)
9. [Advanced Features](#advanced-features)
10. [Callbacks](#callbacks)


## 1. Getting Started {#getting-started}

First, import the SHOTpy module and create an environment.

In [ ]:
import sys
import os

# Add current directory to Python path to find SHOTpy module
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())

import SHOTpy

# Create a SHOT solver and get its environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()
print("SHOT solver and environment created successfully!")

## 2. Creating Variables {#creating-variables}

Variables are the fundamental building blocks of optimization problems. SHOTpy supports:
- **Real** (continuous) variables
- **Integer** variables
- **Binary** variables

In [ ]:
# Create a continuous variable: x ∈ [0, 10]
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)

# Create an integer variable: y ∈ {0, 1, 2, ..., 5}
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)

# Create a binary variable: z ∈ {0, 1}
z = SHOTpy.Variable("z", 2, SHOTpy.VariableType.Binary, 0.0, 1.0)

print(f"Created variables: {x.name}, {y.name}, {z.name}")

## 3. Linear Problems {#linear-problems}

Let's solve a simple linear program (LP):

$$
\begin{align*}
\text{minimize} \quad & 2x + 3y \\
\text{subject to} \quad & x + y \geq 1 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create solver and get environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a new problem
problem = SHOTpy.Problem(env)

# Add variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Create linear objective: minimize 2x + 3y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.LinearTerm(3.0, y))
problem.setObjective(objective)

# Add constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "c1", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Finalize and solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get results
sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

## 4. Mixed-Integer Problems {#mixed-integer-problems}

SHOT excels at solving mixed-integer problems. Here's a simple MIP:

$$
\begin{align*}
\text{maximize} \quad & 3x + 2y \\
\text{subject to} \quad & 2x + y \leq 10 \\
& x, y \in \mathbb{Z}_+
\end{align*}
$$

In [ ]:
# Create a MIP problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Add integer variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Integer, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Objective: maximize 3x + 2y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Maximize)
objective.add(SHOTpy.LinearTerm(3.0, x))
objective.add(SHOTpy.LinearTerm(2.0, y))
problem.setObjective(objective)

# Constraint: 2x + y <= 10
constraint = SHOTpy.LinearConstraint(0, "budget", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.LinearTerm(2.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]}, y = {sol.point[1]}")

## 5. Quadratic Problems {#quadratic-problems}

SHOT can handle quadratic objectives and constraints (QCQP):

$$
\begin{align*}
\text{minimize} \quad & x^2 + y^2 \\
\text{subject to} \quad & x + y \geq 2 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Quadratic objective: x^2 + y^2
objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))  # x^2
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))  # y^2
problem.setObjective(objective)

# Linear constraint: x + y >= 2
constraint = SHOTpy.LinearConstraint(0, "sum_constraint", 2.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Quadratic Constraints

You can also add quadratic constraints, such as circles or ellipses:

In [ ]:
# Problem: minimize x + y subject to x^2 + y^2 <= 4 (inside a circle)
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, -10.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, -10.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Linear objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
objective.add(SHOTpy.LinearTerm(1.0, y))
problem.setObjective(objective)

# Quadratic constraint: x^2 + y^2 <= 4
constraint = SHOTpy.QuadraticConstraint(0, "circle", -SHOTpy.SHOT_DBL_MAX, 4.0)
constraint.add(SHOTpy.QuadraticTerm(1.0, x, x))
constraint.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")
print(f"Distance from origin: {(sol.point[0]**2 + sol.point[1]**2)**0.5:.4f}")

## 6. Nonlinear Problems {#nonlinear-problems}

SHOT is designed for nonlinear problems. You can use various nonlinear expressions:
- Exponential and logarithm: `SHOTpy.exp()`, `SHOTpy.log()`
- Trigonometric: `SHOTpy.sin()`, `SHOTpy.cos()`, `SHOTpy.tan()`
- Square root: `SHOTpy.sqrt()`
- Power functions: `x**n`
- Arithmetic operators: `+`, `-`, `*`, `/`

### Example: Geometric Programming

$$
\begin{align*}
\text{minimize} \quad & e^x + e^y \\
\text{subject to} \quad & e^{x+y} \leq 10 \\
& x, y \geq 0.1
\end{align*}
$$

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Nonlinear objective: exp(x) + exp(y)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.exp(x) + SHOTpy.exp(y))
problem.setObjective(objective)

# Nonlinear constraint: exp(x + y) <= 10
constraint = SHOTpy.NonlinearConstraint(0, "nl_constraint", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.exp(x + y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Example: Mixed Terms

You can combine linear, quadratic, and nonlinear terms:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 5.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mixed objective: x + y^2 + log(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))           # Linear term
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))     # Quadratic term
objective.add(SHOTpy.log(x))                        # Nonlinear term
problem.setObjective(objective)

# Constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "bound", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Advanced Nonlinear Expressions

Complex nested expressions are also supported:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 3.14)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 1.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Complex expression: exp(x) * sin(x) + log(y) * cos(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
expr = SHOTpy.exp(x) * SHOTpy.sin(x) + SHOTpy.log(y) * SHOTpy.cos(x)
objective.add(expr)
problem.setObjective(objective)

# Simple constraint
constraint = SHOTpy.LinearConstraint(0, "bound", 0.5, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint)

problem.finalize()
print("\nProblem structure:")
print(problem.toString())

## 7. Reading Problems from Files {#reading-from-files}

SHOT can read optimization problems from various file formats:
- **OSiL** (Optimization Services Instance Language)
- **GAMS** files (.gms)
- Other formats supported by modeling systems

In [ ]:
# Example: Reading from an OSiL file
# Note: Replace the path with an actual OSiL file path

# problem = SHOTpy.Problem(env)
# problem_file = "/path/to/problem.osil"
# 
# # Read the problem from file
# if problem.importModel(problem_file):
#     print(f"Successfully loaded problem from {problem_file}")
#     problem.finalize()
#     
#     solver = SHOTpy.Solver(env, problem)
#     solver.solveProblem()
#     
#     print(f"Objective value: {solver.getPrimalObjectiveValue():.4f}")
# else:
#     print("Failed to load problem file")

print("See commented code above for file loading example")

## 8. Configuring Solver Settings {#solver-settings}

SHOT provides extensive configuration options through its settings system.

### Accessing and Modifying Settings

In [ ]:
# Create solver and environment first
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a simple problem
problem = SHOTpy.Problem(env)
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)

objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
solver.setProblem(problem)

# Get current values
print("Current Settings:")
time_limit = solver.getDoubleSetting("TimeLimit", "Termination")
print(f"  Time limit: {time_limit} seconds")

abs_gap = solver.getDoubleSetting("ObjectiveGap.Absolute", "Termination")
print(f"  Absolute gap tolerance: {abs_gap}")

# Modify settings
print("\nModifying settings...")
solver.updateSetting("TimeLimit", "Termination", 300.0)  # 5 minutes
solver.updateSetting("ObjectiveGap.Absolute", "Termination", 1e-6)

# Verify changes
new_time = solver.getDoubleSetting("TimeLimit", "Termination")
new_gap = solver.getDoubleSetting("ObjectiveGap.Absolute", "Termination")
print(f"  New time limit: {new_time} seconds")
print(f"  New absolute gap: {new_gap}")

### Common Settings

Here are some frequently used settings:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)
problem.finalize()

solver.setProblem(problem)

# Termination criteria
solver.updateSetting("TimeLimit", "Termination", 600.0)              # Max time in seconds
solver.updateSetting("ObjectiveGap.Absolute", "Termination", 1e-5)   # Absolute gap
solver.updateSetting("ObjectiveGap.Relative", "Termination", 1e-4)   # Relative gap

# Output settings
solver.updateSetting("Console.LogLevel", "Output", 1)                # 0=off, 1=summary, 2=detailed

# MIP solver selection (convert enum to int)
solver.updateSetting("MIP.Solver", "Dual", int(SHOTpy.MIPSolver.Gurobi))  # or Cplex, Cbc, Highs

# NLP solver selection
solver.updateSetting("FixedInteger.Solver", "Primal", int(SHOTpy.PrimalNLPSolver.Ipopt))  # or GAMS

print("Settings configured successfully!")

# Verify some settings
print(f"Time limit: {solver.getDoubleSetting('TimeLimit', 'Termination')} seconds")
print(f"Log level: {solver.getIntSetting('Console.LogLevel', 'Output')}")

## 9. Advanced Features {#advanced-features}

### Monomial and Signomial Terms

For geometric programming, you can use signomial terms:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Create a signomial element: x^2 * y^(-1)
element1 = SHOTpy.SignomialElement(x, 2.0)
element2 = SHOTpy.SignomialElement(y, -1.0)

# Create signomial term: 3.0 * x^2 * y^(-1) - coefficient first, then elements
signomial = SHOTpy.SignomialTerm(3.0, [element1, element2])

# Alternative: Create monomial using variable-power pairs directly
monomial = SHOTpy.SignomialTerm(2.5, [(x, 2.0), (y, 3.0)])

# Use in constraint
constraint = SHOTpy.NonlinearConstraint(0, "signomial_constraint", -SHOTpy.SHOT_DBL_MAX, 100.0)
constraint.add(signomial)
problem.addConstraint(constraint)

# Simple objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
print("Problem with signomial terms created")
print(problem.toString())

### Examining Solution Details

In [ ]:
# Solve a simple problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.setObjective(objective)

constraint = SHOTpy.LinearConstraint(0, "sum", 3.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get detailed results
print("Solution Details:")
sol = solver.getPrimalSolution()
print(f"  Primal objective: {sol.objValue:.6f}")
print(f"  Dual bound: {solver.getCurrentDualBound():.6f}")

solutions = solver.getPrimalSolutions()
print(f"  Number of solutions found: {len(solutions)}")

if len(solutions) > 0:
    print(f"  Best solution: x = {solutions[0].point[0]:.4f}, y = {solutions[0].point[1]:.4f}")

# Get problem name (if set)
print(f"  Problem name: {problem.name if hasattr(problem, 'name') else 'N/A'}")

### Checking Problem Properties

In [ ]:
# Create a problem with various features
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mix of linear, quadratic, and nonlinear terms
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
objective.add(SHOTpy.log(x))
problem.setObjective(objective)

constraint1 = SHOTpy.LinearConstraint(0, "linear", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint1.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint1)

constraint2 = SHOTpy.NonlinearConstraint(1, "nonlinear", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint2.add(SHOTpy.exp(x))
problem.addConstraint(constraint2)

problem.finalize()

# Display problem structure with indicators
print("Problem Structure:")
print(problem.toString())
print("\nIndicators legend:")
print("  L = Linear terms")
print("  Q = Quadratic terms")
print("  S = Signomial terms (power functions)")
print("  M = Monomial terms")
print("  E = General nonlinear expressions")

## 10. Callbacks {#callbacks}

SHOTpy exposes SHOT's callback system via `solver.registerCallback(event, fn)`. Six event types are available through `SHOTpy.EventType`:

| Event type | Callback signature | Description |
|---|---|---|
| `NewPrimalSolution` | `fn(data) -> None` | Called every time a new primal (incumbent) solution is found |
| `PrimalSolutionCandidateSelection` | `fn(data) -> None` | Called when a primal candidate is being evaluated |
| `UserTerminationCheck` | `fn(data) -> bool \| None` | Return `True` to stop the solver early; `None`/`False` continues |
| `ExternalDualBound` | `fn(data) -> float \| None` | Return a tighter dual bound, or `None` to skip |
| `ExternalPrimalSolution` | `fn(data) -> list[float] \| None` | Return a feasible primal point to inject, or `None` to skip |
| `ExternalHyperplaneSelection` | `fn(data) -> list[ExternalHyperplane] \| None` | Supply custom supporting hyperplane cuts |

Each callback receives a typed data object (e.g. `PrimalSolutionCallbackData`, `TerminationCallbackData`) with fields like `iterationNumber`, `currentDualBound`, `relativeGap`, and `solutionStatistics`.


In [ ]:
import SHOTpy

# ── Build the ex1223b MINLP ────────────────────────────────────────────────
solver = SHOTpy.Solver()
env    = solver.getEnvironment()

problem = SHOTpy.Problem(env)
problem.name = "ex1223b_callbacks"

x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,   0.0, 10.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Real,   0.0, 10.0)
x3 = SHOTpy.Variable("x3", 2, SHOTpy.VariableType.Real,   0.0, 10.0)
b4 = SHOTpy.Variable("b4", 3, SHOTpy.VariableType.Binary, 0.0, 1.0)
b5 = SHOTpy.Variable("b5", 4, SHOTpy.VariableType.Binary, 0.0, 1.0)
b6 = SHOTpy.Variable("b6", 5, SHOTpy.VariableType.Binary, 0.0, 1.0)
b7 = SHOTpy.Variable("b7", 6, SHOTpy.VariableType.Binary, 0.0, 1.0)
for v in [x1, x2, x3, b4, b5, b6, b7]:
    problem.addVariable(v)

obj_expr = ((b4 - 1.0)**2 + (b5 - 2.0)**2 + (b6 - 1.0)**2
            - SHOTpy.log(1.0 + b7)
            + (x1 - 1.0)**2 + (x2 - 2.0)**2 + (x3 - 3.0)**2)
problem.setObjective(
    SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize,
                                      obj_expr, 0.0))

lt1 = SHOTpy.LinearTerms()
for v in [x1, x2, x3, b4, b5, b6]:
    lt1.add(SHOTpy.LinearTerm(1.0, v))
problem.addConstraint(SHOTpy.LinearConstraint(0, "e1", lt1, SHOTpy.SHOT_DBL_MIN, 5.0))

problem.addConstraint(SHOTpy.NonlinearConstraint(
    1, "e2", b6**2 + x1**2 + x2**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 5.5))

for idx, (va, vb, rhs, name) in enumerate(
        [(x1, b4, 1.2, "e3"), (x2, b5, 1.8, "e4"),
         (x3, b6, 2.5, "e5"), (x1, b7, 1.2, "e6")], start=2):
    lt = SHOTpy.LinearTerms()
    lt.add(SHOTpy.LinearTerm(1.0, va)); lt.add(SHOTpy.LinearTerm(1.0, vb))
    problem.addConstraint(SHOTpy.LinearConstraint(idx, name, lt, SHOTpy.SHOT_DBL_MIN, rhs))

problem.addConstraint(SHOTpy.NonlinearConstraint(
    6, "e7", b5**2 + x2**2, SHOTpy.SHOT_DBL_MIN, 1.64))
problem.addConstraint(SHOTpy.NonlinearConstraint(
    7, "e8", b6**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.25))
problem.addConstraint(SHOTpy.NonlinearConstraint(
    8, "e9", b5**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.64))
problem.finalize()

# ── Example 1: track every new primal solution ────────────────────────────
primal_log = []

def on_new_primal(data):
    primal_log.append({
        "iter":      data.iterationNumber,
        "obj":       data.objectiveValue,
        "dual":      data.currentDualBound,
        "gap (%)":   round(data.relativeGap * 100, 4),
    })

solver.registerCallback(SHOTpy.EventType.NewPrimalSolution, on_new_primal)

# ── Example 2: stop after the relative gap is tight enough ───────────────
def early_stop(data):
    # Terminate once the gap drops below 1 %
    return data.relativeGap < 0.01

solver.registerCallback(SHOTpy.EventType.UserTerminationCheck, early_stop)

# ── Suppress console output and solve ────────────────────────────────────
solver.updateSetting("Console.LogLevel", "Output", 6)  # 6 = Off
solver.setProblem(problem)
solver.solveProblem()

# ── Results ───────────────────────────────────────────────────────────────
print(f"Status : {solver.getModelReturnStatus()}")
print(f"Optimal: {solver.getPrimalBound():.6f}  (known optimum ≈ 4.5796)")
print(f"\nPrimal solution improvements ({len(primal_log)} updates):")
print(f"  {'iter':>5}  {'objective':>12}  {'dual bound':>12}  {'gap %':>8}")
for row in primal_log:
    print(f"  {row['iter']:>5}  {row['obj']:>12.6f}  {row['dual']:>12.6f}  {row['gap (%)']:>8.4f}")

# ── Example 3: inspect an ExternalHyperplane object ──────────────────────
hp = SHOTpy.ExternalHyperplane()
hp.variableIndexes      = [0, 2]        # x1 and x3
hp.variableCoefficients = [1.5, -0.5]   # 1.5*x1 - 0.5*x3 <= rhs
hp.rhsValue             = 3.0
hp.isGlobal             = True
hp.source               = SHOTpy.HyperplaneSource.External
hp.description          = "my_custom_cut"

print("\nExample ExternalHyperplane:")
print(f"  indexes : {hp.variableIndexes}")
print(f"  coeffs  : {hp.variableCoefficients}")
print(f"  rhs     : {hp.rhsValue}")
print(f"  global  : {hp.isGlobal}")
print(f"  source  : {hp.source}")


### External Hyperplane Callback: Solving `shot_ex_jogo`

This example demonstrates the `ExternalHyperplaneSelection` callback by solving a small MINLP
whose problem data comes from `test/data/shot_ex_jogo.gms`.  The problem is constructed
directly through the Python API.

$$
\min_{x_1,\,x_2} \; -x_1 - x_2
$$
subject to

$$
2x_1 - 3x_2 \le 2, \qquad x_1 \in [1,20],\; x_2 \in \mathbb{Z} \cap [1,20]
$$
$$
0.15(x_1-8)^2 + 0.1(x_2-6)^2 + \frac{0.025\,e^{x_1}}{x_2^2} \le 5
$$
$$
\frac{1}{x_1} + \frac{1}{x_2} - \sqrt{x_1}\,\sqrt{x_2} \le -4
$$

All built-in SHOT hyperplane generation is disabled (`CutStrategy = OnlyExternal`).
The callback computes a linearisation cut (supporting hyperplane) for the most-violated
nonlinear constraint at every MIP node and returns it to SHOT as an `ExternalHyperplane`.


In [ ]:
import SHOTpy

# ── Create solver and configure settings ─────────────────────────────────────
solver = SHOTpy.Solver()
env    = solver.getEnvironment()

solver.updateSetting("Console.LogLevel", "Output", 2)

# Disable all built-in hyperplane generation; cuts come exclusively from the callback
solver.updateSetting("Convexity.AssumeConvex", "Model", True)   # True\n
solver.updateSetting("Reformulation.Constraint.PartitionQuadraticTerms", "Model", 2)  # ES_PartitionNonlinearSums::Never
solver.updateSetting("CutStrategy", "Dual",  2)    # ES_HyperplaneCutStrategy::OnlyExternal
solver.updateSetting("TreeStrategy", "Dual",  0)    # ES_TreeStrategy::MultiTree

# ── Build the shot_ex_jogo problem via the Python API ─────────────────────────
#
#   min  -x1 - x2
#   s.t. 2*x1 - 3*x2                            <=  2   (l,  linear)
#        0.15*(x1-8)^2 + 0.1*(x2-6)^2
#          + 0.025*exp(x1)/x2^2                 <=  5   (c1, nonlinear)
#        1/x1 + 1/x2 - sqrt(x1)*sqrt(x2)        <= -4   (c2, nonlinear / signomial)
#   x1 in [1, 20]  (continuous)
#   x2 in [1, 20]  (integer)

problem      = SHOTpy.Problem(env)
problem.name = "shot_ex_jogo"

x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,    1.0, 20.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Integer, 1.0, 20.0)
problem.addVariable(x1)
problem.addVariable(x2)

# Objective: minimize -x1 - x2
obj = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
obj.add(SHOTpy.LinearTerm(-1.0, x1))
obj.add(SHOTpy.LinearTerm(-1.0, x2))
problem.setObjective(obj)

# Constraint l: 2*x1 - 3*x2 <= 2
lt_l = SHOTpy.LinearTerms()
lt_l.add(SHOTpy.LinearTerm( 2.0, x1))
lt_l.add(SHOTpy.LinearTerm(-3.0, x2))
problem.addConstraint(SHOTpy.LinearConstraint(0, "l", lt_l, SHOTpy.SHOT_DBL_MIN, 2.0))

# Constraint c1: 0.15*(x1-8)^2 + 0.1*(x2-6)^2 + 0.025*exp(x1)/x2^2 <= 5
c1 = SHOTpy.NonlinearConstraint(1, "c1", SHOTpy.SHOT_DBL_MIN, 5.0)
c1.add(0.15 * (x1 - 8.0)**2 + 0.1 * (x2 - 6.0)**2 + 0.025 * SHOTpy.exp(x1) / x2**2)
problem.addConstraint(c1)

# Constraint c2: 1/x1 + 1/x2 - x1^0.5 * x2^0.5 <= -4
# Expressed as signomial terms: x1^(-1) + x2^(-1) - x1^0.5 * x2^0.5
c2 = SHOTpy.NonlinearConstraint(2, "c2", SHOTpy.SHOT_DBL_MIN, -4.0)
c2.add(SHOTpy.SignomialTerm( 1.0, [(x1, -1.0)]))              # 1/x1
c2.add(SHOTpy.SignomialTerm( 1.0, [(x2, -1.0)]))              # 1/x2
c2.add(SHOTpy.SignomialTerm(-1.0, [(x1, 0.5), (x2, 0.5)]))   # -sqrt(x1)*sqrt(x2)
problem.addConstraint(c2)

# Keep a reference to the nonlinear constraints for gradient computations in the callback
nonlinear_constraints = [c1, c2]

problem.finalize()

# Pass the same problem as both original and reformulated to bypass internal reformulation
solver.setProblem(problem, problem)

print("Problem created:")
print(problem.toString())

# ── Register external hyperplane callback ─────────────────────────────────────
#
# At every MIP node SHOT calls this function with a list of solution points.
# For each point we linearise the most-violated nonlinear constraint at that
# point and return the resulting cut as an ExternalHyperplane.
#
# Supporting hyperplane for  g(x) <= b  at  x0:
#   grad g(x0) . x  <=  grad g(x0) . x0  -  (g(x0) - b)
#                   <=  sum_i grad_i * x0_i  -  violation

cut_log = []   # record (iteration, constraint name, violation) for display

def generate_hyperplanes(data):
    hyperplanes = []

    if not data.solutionPoints or data.iterationNumber == 0:
        return hyperplanes

    reform_problem = data.reformulatedProblem

    for sol_point in data.solutionPoints:
        dev_idx = sol_point.maxDeviation.index

        # Index must be valid inside the nonlinear-constraint list
        if dev_idx < 0 or dev_idx >= len(reform_problem.nonlinearConstraints):
            continue

        violation = sol_point.maxDeviation.value
        if violation <= 0.0:
            continue   # constraint is satisfied at this point

        # Looks up by the constraint's own .index attribute
        constraint = next(
            (nlc for nlc in reform_problem.nonlinearConstraints if nlc.index == dev_idx), None)
        
        point      = sol_point.point

        # Sparse gradient dict {var_index: coefficient}
        gradient = constraint.calculateGradient(point)

        if not gradient:
            continue

        var_indices  = list(gradient.keys())
        var_coeffs   = list(gradient.values())

        # rhs = grad . x0 - violation
        rhs = sum(gradient[i] * point[i] for i in var_indices) - violation

        hp                      = SHOTpy.ExternalHyperplane()
        hp.variableIndexes      = var_indices
        hp.variableCoefficients = var_coeffs
        hp.rhsValue             = rhs
        hp.isGlobal             = True
        hp.source               = SHOTpy.HyperplaneSource.External
        hp.description          = f"ext_hp_iter{data.iterationNumber}_c{dev_idx}"

        hyperplanes.append(hp)
        cut_log.append((data.iterationNumber, constraint.name, round(violation, 6)))

        break   # one cut per callback invocation is sufficient

    return hyperplanes

solver.registerCallback(SHOTpy.EventType.ExternalHyperplaneSelection, generate_hyperplanes)

# ── Solve ──────────────────────────────────────────────────────────────────────
print("\nSolving with external hyperplanes only...\n")
solver.solveProblem()

# ── Results ───────────────────────────────────────────────────────────────────
print(f"\nStatus : {solver.getModelReturnStatus()}")

if solver.getPrimalSolutions():
    sol = solver.getPrimalSolution()
    print(f"Objective value : {sol.objValue:.6f}")
    print(f"x1 = {sol.point[0]:.6f}  (continuous)")
    print(f"x2 = {sol.point[1]:.6f}  (integer)")
else:
    print("No primal solution found.")

print(f"\nExternal cuts added: {len(cut_log)}")
if cut_log:
    print(f"  {'iter':>6}  {'constraint':>12}  {'violation':>12}")
    for (it, cname, viol) in cut_log[:10]:
        print(f"  {it:>6}  {cname:>12}  {viol:>12.6f}")
    if len(cut_log) > 10:
        print(f"  ... ({len(cut_log) - 10} more)")


## Summary

This tutorial covered:

1. **Basic setup** - Creating environments and problems
2. **Variables** - Real, integer, and binary variables
3. **Linear programming** - LP formulation and solving
4. **Mixed-integer programming** - MIP problems
5. **Quadratic programming** - QP and QCQP problems
6. **Nonlinear programming** - Using exp, log, sin, cos, and other functions
7. **File I/O** - Reading problems from OSiL and GAMS files
8. **Settings** - Configuring solver parameters
9. **Advanced features** - Signomial terms, solution details, and problem properties
10. **Callbacks** - Reacting to solver events, early termination, and custom hyperplane injection

### Additional Resources

- **SHOT Documentation**: https://shotsolver.dev
- **SHOT Repository**: https://github.com/coin-or/SHOT
- **Test Files**: See the `test/python/` directory for more examples

### Tips for Best Results

1. Always call `problem.finalize()` before creating a solver
2. Use appropriate variable bounds to help the solver
3. Configure MIP and NLP subsolvers based on problem characteristics
4. Adjust termination criteria (time limit, gap tolerances) as needed
5. Check solution status and bounds after solving
